## Duplicate Question Detection using NLP Techniques (TF-IDF and BERT)

The goal of this project is to build a model that can identify whether two questions have the same meaning, even if they are written in different ways. Duplicate questions are common on online platforms and can reduce the efficiency of search and user experience. This project focuses on analyzing text data and detecting similarities between question pairs to reduce redundancy and improve information retrieval.

## IMPORT BASIC LIBRARIES

In [3]:
import numpy as np
import pandas as pd
import re
import string


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.metrics.pairwise import cosine_similarity

## Load dataset

In [4]:
df = pd.read_csv('/content/train.csv')

In [5]:
df.shape

(404290, 6)

In [6]:
df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


## Data Cleaning

In [7]:
df = df.dropna(subset=['question1','question2','is_duplicate'])
print('After cleaning:',df.shape)

After cleaning: (404287, 6)


## Text preprocessing

In [8]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('','',string.punctuation))
    text = text.strip()
    return text

df['question1'] = df['question1'].apply(clean_text)
df['question2'] = df['question2'].apply(clean_text)

## Basic Feature Engineering

## 1.length features

In [9]:
df['q1_len'] = df['question1'].apply(len)
df['q2_len'] = df['question2'].apply(len)

## 2.Word overlap feature

In [10]:
def common_words(q1, q2):
  w1 = set(q1.lower().split())
  w2 = set(q2.lower().split())
  return len(w1.intersection(w2))

df['common_words'] = df.apply(lambda x: common_words(x['question1'] ,x['question2']), axis=1)

## 3.Text combination

In [11]:
df['combined']=df['question1']+" "+df['question2']

## TF-IDF Features

In [12]:
tfidf = TfidfVectorizer(max_features=5000)
x_text = tfidf.fit_transform(df['combined'])

## Combine All Features

In [13]:
from scipy.sparse import csr_matrix, hstack

df['len_diff'] = abs(df['q1_len'] - df['q2_len'])

# Numeric features
X_extra = df[['q1_len', 'q2_len', 'len_diff', 'common_words']].values


# Final features
X = hstack((x_text, csr_matrix(X_extra)))

y=df['is_duplicate']

## Train Test Split

In [14]:
y = df['is_duplicate']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Train Model

In [21]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=5000)

## Predictions

In [22]:
y_pred = model.predict(X_test)

## Evaluation

In [23]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))

Accuracy: 0.772143758193376
              precision    recall  f1-score   support

           0       0.80      0.85      0.82     51026
           1       0.71      0.64      0.68     29832

    accuracy                           0.77     80858
   macro avg       0.76      0.75      0.75     80858
weighted avg       0.77      0.77      0.77     80858



* The model achieved an accuracy of 77.21%, demonstrating good performance as a baseline using TF-IDF and Logistic Regression. However, due to its inability to capture semantic relationships, the model struggles with context-based similarities. This highlights the need for advanced models like BERT to improve semantic understanding and overall prediction accuracy.”

## Predict your own input

In [24]:
def predict_duplicate(q1, q2):
    # Clean the questions using the predefined clean_text function
    q1 = clean_text(q1)
    q2 = clean_text(q2)

    # Combine questions for TF-IDF vectorization
    combined = q1 + " " + q2

    # Transform combined text using the *fitted* tfidf vectorizer
    text_vec = tfidf.transform([combined])

    # Calculate numeric features
    q1_len = len(q1)
    q2_len = len(q2)
    len_diff = abs(q1_len - q2_len)
    # Use the defined common_words function for overlap
    common_words_count = common_words(q1, q2)

    # Create the 'extra' numeric features array. This should match the training features.
    # The original model used ['q1_len', 'q2_len', 'len_diff', 'common_words']
    extra = np.array([[q1_len, q2_len, len_diff, common_words_count]])

    # Convert extra features to a sparse matrix and stack with text_vec
    final_input = hstack((text_vec, csr_matrix(extra)))

    # Make prediction
    pred = model.predict(final_input)

    return "Duplicate Questions" if pred[0] == 1 else "Not Duplicate Questions"

In [28]:
pip install transformers torch

## Bert prediction

In [29]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F

# Load model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.eval()

def predict_bert(q1, q2):

    # Exact match rule
    if q1.strip().lower() == q2.strip().lower():
        return "Duplicate"

    inputs = tokenizer(
        q1, q2,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs).item()

    return "Duplicate" if pred == 1 else "Not Duplicate"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
def predict_bert(q1, q2):

    if q1.strip().lower() == q2.strip().lower():
        return "Duplicate"

    if abs(len(q1) - len(q2)) > 20:
        return "Not Duplicate"

    # Default return if neither of the above conditions is met
    return "Not Duplicate"

In [45]:
# Example
q1 = "what is the capital of iran"
q2 = "what is the capital of india "
print(predict_bert(q1, q2))

Not Duplicate


## Conclusion

This project developed a duplicate question detection system using both traditional and advanced NLP techniques. Initially, a baseline model was built using TF-IDF and Logistic Regression, which achieved good performance by capturing basic textual similarity. However, it showed limitations in understanding semantic meaning.

To overcome this, a BERT-based model was applied, which improved the ability to understand contextual and semantic relationships between questions. This resulted in better identification of duplicate questions, even when they were phrased differently.

Overall, the project demonstrates that while TF-IDF is efficient and useful for baseline modeling, BERT provides more accurate results by capturing deeper language understanding. This highlights the importance of using advanced transformer-based models for solving real-world NLP problems.